### Task 1

In [45]:
import math
import pandas as pd

insurance_df = pd.read_csv('Data/Insurance.csv')
insurance_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


1. 7 Columns, 1338 rows
2. Each column represents a quality about a person. Each row is a specific person.
3. The label column is `charges`
4. The rest of the columns are features: age, sex, bmi, children, smoker, region

In [46]:
insurance_df['bmi'].describe()

count    1338.000000
mean       30.663397
std         6.098187
min        15.960000
25%        26.296250
50%        30.400000
75%        34.693750
max        53.130000
Name: bmi, dtype: float64

### Task 2

In [47]:
brian_guess = 3000
jesse_guess = 20000

### Task 3

In [48]:
mean_baseline = round(insurance_df['charges'].mean(), 2)
print(f'Mean Baseline: ${mean_baseline}')
insurance_df['charge_baseline'] = mean_baseline

# Our guesses were not very close. Almost the same distance, one higher and one lower

Mean Baseline: $13270.42


### Task 4

In [49]:
insurance_df['baseline_error'] = abs(insurance_df['charges'] - insurance_df['charge_baseline'])
baseline_mae = round(insurance_df['baseline_error'].mean(),2)
print(f'Baseline MAE: ${baseline_mae}')

# Our baseline MAE is $9000 which means that this very basic predictive model is generally accurate
#      within $9000.

Baseline MAE: $9091.13


In [50]:
insurance_df.head()

,age,sex,bmi,children,smoker,region,charges,charge_baseline,baseline_error
0,19,female,27.900,0,yes,southwest,16884.92400,13270.42,3614.50400
1,18,male,33.770,1,no,southeast,1725.55230,13270.42,11544.86770
2,28,male,33.000,3,no,southeast,4449.46200,13270.42,8820.95800
3,33,male,22.705,0,no,northwest,21984.47061,13270.42,8714.05061
4,32,male,28.880,0,no,northwest,3866.85520,13270.42,9403.56480


### Task 5

In [51]:
baseline_mse = insurance_df['baseline_error'].apply(lambda x: x**2).mean()
print(f'Baseline MSE: {baseline_mse}')

baseline_rmse = round(math.sqrt(baseline_mse),2)
print(f'Baseline RMSE: ${baseline_rmse}')


# Units for MAE and RMSE are Dollars (dolla dolla bills yall)
# RMSE is prefered when larger errors are more damaging, so we penalize them.

Baseline MSE: 146542766.49355304
Baseline RMSE: $12105.48


### Task 6

In [52]:
smoker_mean = insurance_df[insurance_df["smoker"] == 'yes']['charges'].mean()
non_smoker_mean = insurance_df[insurance_df["smoker"] == 'no']['charges'].mean()

insurance_df["smoker_mean"] = insurance_df["smoker"].map({'yes': smoker_mean, 'no': non_smoker_mean})

error = [abs(c - a) for c, a in zip(insurance_df["charges"], insurance_df["smoker_mean"])]

grouped_mae = round(sum(error) / len(error), 2)

print(f'Grouped MAE: ${grouped_mae}')

Grouped MAE: $5662.09


### Task 7

In [53]:
x = insurance_df.drop(columns=["charges", "charge_baseline", "baseline_error", "smoker_mean"])
y = insurance_df["charges"]

#So we can test how close we get to y by predicting the charges with x

### Task 8

In [58]:
x_train = x.iloc[:int(len(x) * .8)]
y_train = y.iloc[:int(len(x) * .8)]
x_test = x.iloc[int(len(x) * .8):]
y_test = y.iloc[int(len(x) * .8):]

print("x_train rows:", len(x_train))
print("y_train rows:", len(y_train))
print("x_test rows:", len(x_test))
print("y_test rows:", len(y_test))


#It can memorize the specific data too well. We won't know if it's giving good predictions or it just memorized the data.
#Overfitting

x_train rows: 1070
y_train rows: 1070
x_test rows: 268
y_test rows: 268


### Task 9

In [55]:
mean_baseline = round(y_train.mean(), 2)
test_baseline_mae = round(abs(y_test - mean_baseline).mean(),2)
xy_train = x_train.merge(y_train, left_on=x_train.index, right_on=y_train.index)
smoker_mean = round(xy_train[xy_train["smoker"] == "yes"]["charges"].mean(), 2)
non_smoker_mean = round(xy_train[xy_train["smoker"] == "no"]["charges"].mean(), 2)

xy_test = x_test.merge(y_test, left_on=x_test.index, right_on=y_test.index)

def get_group_mae(df):
    if df["smoker"] == "yes":
       return (abs(df["charges"] - smoker_mean))
    else:
        return (abs(df["charges"] - non_smoker_mean))

test_grouped_mae = round(xy_test.apply(get_group_mae, axis=1).mean(), 2)

print(f'Test Baseline MAE: ${test_baseline_mae}')
print(f'Test Grouped MAE: ${test_grouped_mae}')


# 

Test Baseline MAE: $9197.53
Test Grouped MAE: $5504.59


Baseline MAE:       $9091.13
Grouped MAE:        $5662.09

Test Baseline MAE:  $9197.53
Test Grouped MAE:   $5504.59

The test grouped MAE is lower, which means that it is more accurate.

### Task 10

This dataset contains information about insurance customers and their charges. We attempted to predict what the total charges would be based on whether or not the individual was a smoker. Our guesses about the mean were not very close. Almost the same distance, one higher and one lower. The grouped baseline was the most accurate. MAE is the average error for each prediction over the whole dataset. MSE makes larger errors more apparent. RMSE keeps the extra weight of large errors from MSE but puts the units back into something more understandable for people. Splitting train/test data keeps the model from overfitting - learning the specifics of that dataset too well and not learning the real patterns.

## Advanced Tasks

### Advanced Task 1

In [57]:
xy_test['bmi_group'] = pd.cut(xy_test['bmi'], bins=3, labels=('underweight','healthy','overweight'))
bmi_smoker_avg = pd.crosstab(xy_test['smoker'], xy_test['bmi_group'], xy_test['charges'], aggfunc='mean')

# print(bmi_smoker_avg)
smoker = 'no'
bmi_group = 'underweight'
# print(bmi_smoker_avg.loc[smoker,bmi_group])


def get_bmi_smoker_error(row):
    # if row['smoker'] == 'yes' and row['bmi_group'] == 'underweight':
    smoker = row['smoker']
    bmi_group = row['bmi_group']
    grouped_avg = bmi_smoker_avg.loc[smoker, bmi_group]
    return abs(row['charges'] - grouped_avg)

bmi_smoker_mae = round(xy_test.apply(get_bmi_smoker_error, axis=1).mean(), 2)
print(f'MAE when using Smoking and BMI as features: ${bmi_smoker_mae}')

MAE when using Smoking and BMI as features: $4426.59
